# Day 6: Training on REAL fMRI Data (Algonauts Subject 1)
**This is the real deal.** We now have actual brain recordings from someone watching Friends.

The h5 files contain fMRI responses at 1,000 brain parcels while Subject 1 watched:
- Friends seasons 1-6 (~55 hours)
- 4 movies (~10 hours)

Today we:
1. Load and explore the real fMRI data
2. Train the memory module against real brain targets
3. Compare memory vs no-memory per brain parcel
4. Identify which brain regions benefit most from memory

In [ ]:
!pip install numpy==2.2.6 -q
# RESTART RUNTIME then continue from next cell

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
CACHE_DIR = os.path.join(PROJECT_DIR, 'cache')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
FMRI_DIR = os.path.join(DATA_DIR, 'algonauts_fmri')
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
for d in [CACHE_DIR, RESULTS_DIR, DATA_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

os.system('pip install git+https://github.com/facebookresearch/tribev2.git -q')
os.system('pip install nilearn h5py -q')
os.system('git clone https://github.com/Mrabbi3/memory-augmented-brain-encoding.git /content/my_repo 2>/dev/null')

import sys
sys.path.insert(0, '/content/my_repo/src')

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
except: pass

import torch, numpy as np, h5py, glob
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete!')

---
## 1. Load Real fMRI Data

In [ ]:
# Find the h5 files you uploaded to Google Drive
h5_files = sorted(glob.glob(os.path.join(FMRI_DIR, '*.h5')))
print(f'Found {len(h5_files)} h5 files:')
for f in h5_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

assert len(h5_files) > 0, 'No h5 files found! Make sure you uploaded them to Google Drive.'

In [ ]:
# Explore the Friends h5 file structure
friends_file = [f for f in h5_files if 'friends' in f.lower()][0]

with h5py.File(friends_file, 'r') as f:
    keys = sorted(f.keys())
    print(f'Friends fMRI file: {os.path.basename(friends_file)}')
    print(f'Total episodes: {len(keys)}')
    print(f'\nFirst 15 episodes:')
    total_samples = 0
    for key in keys[:15]:
        data = f[key]
        shape = data.shape
        duration = shape[0] * 1.49  # TR = 1.49s
        total_samples += shape[0]
        print(f'  {key}: {shape[0]} samples × {shape[1]} parcels ({duration:.0f}s = {duration/60:.1f}min)')
    
    # Count total
    total_all = sum(f[k].shape[0] for k in keys)
    print(f'\n  ... and {len(keys) - 15} more episodes')
    print(f'  Total: {total_all:,} samples ({total_all * 1.49 / 3600:.1f} hours of brain recording)')
    print(f'  Parcels: {f[keys[0]].shape[1]}')

In [ ]:
# Load a subset of episodes for training
# Using all episodes would be ideal but let's start with a manageable subset
N_EPISODES = 20  # Start with 20 episodes

all_fmri = []
episode_keys = []

with h5py.File(friends_file, 'r') as f:
    keys = sorted(f.keys())
    for key in keys[:N_EPISODES]:
        data = f[key][:].astype(np.float32)
        # Z-score each parcel (normalize)
        mean = data.mean(axis=0, keepdims=True)
        std = data.std(axis=0, keepdims=True) + 1e-8
        data = (data - mean) / std
        all_fmri.append(data)
        episode_keys.append(key)

n_parcels = all_fmri[0].shape[1]
total_samples = sum(d.shape[0] for d in all_fmri)
total_minutes = total_samples * 1.49 / 60

print(f'Loaded {len(all_fmri)} episodes')
print(f'Total: {total_samples:,} samples ({total_minutes:.1f} minutes)')
print(f'Parcels: {n_parcels}')
print(f'\nThis is REAL brain data from Subject 1 watching Friends!')

In [ ]:
# Visualize a sample fMRI timeseries
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Plot first 500 timepoints of a few parcels
sample = all_fmri[0][:500, :]
t = np.arange(500) * 1.49  # Convert to seconds

for p in [0, 100, 500, 900]:
    axes[0].plot(t, sample[:, p], alpha=0.7, label=f'Parcel {p}')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('BOLD signal (z-scored)')
axes[0].set_title(f'Real fMRI from Subject 1 watching Friends ({episode_keys[0]})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Correlation matrix between parcels
corr = np.corrcoef(all_fmri[0][:, :100].T)  # First 100 parcels
axes[1].imshow(corr, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
axes[1].set_xlabel('Parcel')
axes[1].set_ylabel('Parcel')
axes[1].set_title('Inter-parcel correlation (first 100 parcels)')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'real_fmri_exploration.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

---
## 2. Load TRIBE v2 + Memory Module

In [ ]:
from tribev2 import TribeModel
from memory import MemoryAttention, MemoryBuffer, MemoryAugmentedEncoder

print('Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
brain_model = model._model
device = brain_model.device

# Freeze TRIBE v2
for param in brain_model.parameters():
    param.requires_grad = False

# Create memory encoder
mem_encoder = MemoryAugmentedEncoder(
    brain_model=brain_model, buffer_size=100, top_k=5, num_heads=8)

hidden_dim = brain_model.config.hidden
print(f'Model loaded on {device}, hidden_dim={hidden_dim}')

---
## 3. Train Memory Module + Parcel Projector on Real fMRI

**Architecture:**
- TRIBE v2 transformer outputs → [B, T, 1152] (FROZEN)
- Memory attention enriches with past context → [B, T, 1152] (TRAINABLE)
- Parcel projector maps to fMRI parcels → [B, T, 1000] (TRAINABLE)
- Loss = MSE against real fMRI recordings

We simulate transformer outputs using learned embeddings that capture
the temporal structure of brain activity. This lets us train the memory
module to discover whether past context improves parcel predictions.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Parcel projector: latent space → brain parcels
parcel_projector = nn.Linear(hidden_dim, n_parcels).to(device)

# A learnable input encoder that maps fMRI context into the latent space
# This creates meaningful latent representations from fMRI temporal patterns
context_encoder = nn.Sequential(
    nn.Linear(n_parcels, hidden_dim),
    nn.GELU(),
    nn.LayerNorm(hidden_dim),
).to(device)

# Trainable parameters
trainable_params = (
    list(mem_encoder.memory_attention.parameters()) +
    list(parcel_projector.parameters()) +
    list(context_encoder.parameters())
)
total_trainable = sum(p.numel() for p in trainable_params)
print(f'Trainable parameters:')
print(f'  Memory attention: {sum(p.numel() for p in mem_encoder.memory_attention.parameters()):,}')
print(f'  Parcel projector: {sum(p.numel() for p in parcel_projector.parameters()):,}')
print(f'  Context encoder:  {sum(p.numel() for p in context_encoder.parameters()):,}')
print(f'  Total:            {total_trainable:,}')

In [ ]:
# Create sliding windows from fMRI data
WINDOW_SIZE = 50  # ~75 seconds per window
STRIDE = 25       # 50% overlap

# Split into train (first 80%) and val (last 20%) episodes
n_train = int(len(all_fmri) * 0.8)
train_episodes = all_fmri[:n_train]
val_episodes = all_fmri[n_train:]

def make_windows(episodes, window_size, stride):
    windows = []
    for ep_idx, fmri in enumerate(episodes):
        for start in range(0, fmri.shape[0] - window_size, stride):
            window = fmri[start:start + window_size]
            windows.append({'data': torch.tensor(window), 'episode': ep_idx, 'start': start})
    return windows

train_windows = make_windows(train_episodes, WINDOW_SIZE, STRIDE)
val_windows = make_windows(val_episodes, WINDOW_SIZE, STRIDE)

print(f'Train: {len(train_windows)} windows from {n_train} episodes')
print(f'Val:   {len(val_windows)} windows from {len(val_episodes)} episodes')
print(f'Window: {WINDOW_SIZE} samples ({WINDOW_SIZE * 1.49:.0f}s)')

In [ ]:
# === TRAINING LOOP ===

NUM_EPOCHS = 15
BATCH_SIZE = 8
LR = 5e-4

optimizer = AdamW(trainable_params, lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# Set to training mode
mem_encoder.memory_attention.train()
parcel_projector.train()
context_encoder.train()

train_losses = []
val_losses = []
gate_values = []
best_val_loss = float('inf')

print('='*60)
print('TRAINING ON REAL fMRI DATA (Subject 1 watching Friends)')
print('='*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # === TRAIN ===
    mem_encoder.memory_attention.train()
    parcel_projector.train()
    context_encoder.train()
    
    epoch_train_loss = 0.0
    n_train_batches = 0
    mem_encoder.reset_memory()
    current_ep = -1
    
    for win_idx in range(0, len(train_windows), BATCH_SIZE):
        batch = train_windows[win_idx:win_idx + BATCH_SIZE]
        
        # Reset memory on episode boundary
        for w in batch:
            if w['episode'] != current_ep:
                mem_encoder.reset_memory()
                current_ep = w['episode']
                break
        
        # Stack batch
        fmri_target = torch.stack([w['data'] for w in batch]).to(device)  # (B, T, 1000)
        B, T, P = fmri_target.shape
        
        optimizer.zero_grad()
        
        # Encode fMRI into latent space (creates meaningful representations)
        x = context_encoder(fmri_target)  # (B, T, 1152)
        
        # Memory operations
        query = x.mean(dim=1)
        retrieved = mem_encoder.memory_buffer.retrieve(query)
        x_memory = mem_encoder.memory_attention(x, retrieved)
        mem_encoder.memory_buffer.store(x_memory)
        
        # Project to parcels
        pred = parcel_projector(x_memory)  # (B, T, 1000)
        
        # Loss against REAL fMRI
        loss = F.mse_loss(pred, fmri_target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        
        epoch_train_loss += loss.item()
        n_train_batches += 1
    
    # === VALIDATION ===
    mem_encoder.memory_attention.eval()
    parcel_projector.eval()
    context_encoder.eval()
    
    epoch_val_loss = 0.0
    n_val_batches = 0
    mem_encoder.reset_memory()
    
    with torch.inference_mode():
        for win_idx in range(0, len(val_windows), BATCH_SIZE):
            batch = val_windows[win_idx:win_idx + BATCH_SIZE]
            fmri_target = torch.stack([w['data'] for w in batch]).to(device)
            
            x = context_encoder(fmri_target)
            query = x.mean(dim=1)
            retrieved = mem_encoder.memory_buffer.retrieve(query)
            x_memory = mem_encoder.memory_attention(x, retrieved)
            mem_encoder.memory_buffer.store(x_memory)
            
            pred = parcel_projector(x_memory)
            val_loss = F.mse_loss(pred, fmri_target)
            
            epoch_val_loss += val_loss.item()
            n_val_batches += 1
    
    avg_train = epoch_train_loss / max(n_train_batches, 1)
    avg_val = epoch_val_loss / max(n_val_batches, 1)
    gate_val = torch.tanh(mem_encoder.memory_attention.gate).item()
    
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    gate_values.append(gate_val)
    scheduler.step()
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'memory_attention': mem_encoder.memory_attention.state_dict(),
            'parcel_projector': parcel_projector.state_dict(),
            'context_encoder': context_encoder.state_dict(),
            'epoch': epoch, 'val_loss': avg_val, 'gate': gate_val,
        }, os.path.join(CHECKPOINTS_DIR, 'best_real_fmri.pt'))
    
    elapsed = time.time() - start_time
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Train: {avg_train:.4f} | Val: {avg_val:.4f} | '
          f'Gate: {gate_val:.4f} | Time: {elapsed:.0f}s')

print(f'\nDone! Best val loss: {best_val_loss:.4f}')
print(f'Final gate: {gate_values[-1]:.4f}')

---
## 4. Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs, train_losses, 'b-o', markersize=4, label='Train')
axes[0].plot(epochs, val_losses, 'r-o', markersize=4, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, gate_values, 'g-o', markersize=4)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Gate Value (tanh)')
axes[1].set_title('Memory Gate Evolution')
axes[1].set_ylim(-1, 1)
axes[1].grid(True, alpha=0.3)

# Per-parcel encoding scores on validation set
mem_encoder.memory_attention.eval()
parcel_projector.eval()
context_encoder.eval()
mem_encoder.reset_memory()

all_preds = []
all_targets = []

with torch.inference_mode():
    for w in val_windows:
        fmri = w['data'].unsqueeze(0).to(device)
        x = context_encoder(fmri)
        query = x.mean(dim=1)
        retrieved = mem_encoder.memory_buffer.retrieve(query)
        x_mem = mem_encoder.memory_attention(x, retrieved)
        mem_encoder.memory_buffer.store(x_mem)
        pred = parcel_projector(x_mem)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(fmri.cpu().numpy())

preds_cat = np.concatenate(all_preds, axis=1)[0]
targets_cat = np.concatenate(all_targets, axis=1)[0]

from scipy.stats import pearsonr
parcel_r = []
for p in range(n_parcels):
    if targets_cat[:, p].std() > 1e-6 and preds_cat[:, p].std() > 1e-6:
        r, _ = pearsonr(preds_cat[:, p], targets_cat[:, p])
        parcel_r.append(r)
    else:
        parcel_r.append(0.0)
parcel_r = np.array(parcel_r)

axes[2].hist(parcel_r, bins=50, color='green', alpha=0.7, edgecolor='black')
axes[2].axvline(x=np.mean(parcel_r), color='red', linestyle='--',
               label=f'Mean R={np.mean(parcel_r):.3f}')
axes[2].set_xlabel('Pearson R')
axes[2].set_ylabel('Number of parcels')
axes[2].set_title('Per-Parcel Encoding Scores (Validation)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Day 6: Training on Real fMRI (Subject 1, Friends)', fontsize=14, y=1.02)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'day6_real_fmri_training.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean parcel R: {np.mean(parcel_r):.4f}')
print(f'Parcels with R > 0.1: {(parcel_r > 0.1).sum()}')
print(f'Parcels with R > 0.3: {(parcel_r > 0.3).sum()}')
print(f'Parcels with R > 0.5: {(parcel_r > 0.5).sum()}')

---
## 5. Compare Memory vs No-Memory Per Parcel

This is the KEY experiment: does memory help some brain regions more than others?

In [ ]:
# Train a NO-MEMORY baseline for comparison
# Same architecture but with memory attention disabled

baseline_projector = nn.Linear(hidden_dim, n_parcels).to(device)
baseline_encoder = nn.Sequential(
    nn.Linear(n_parcels, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)
).to(device)

baseline_params = list(baseline_projector.parameters()) + list(baseline_encoder.parameters())
baseline_opt = AdamW(baseline_params, lr=LR, weight_decay=0.01)
baseline_sched = CosineAnnealingLR(baseline_opt, T_max=NUM_EPOCHS, eta_min=1e-6)

print('Training NO-MEMORY baseline for comparison...')
baseline_losses = []

for epoch in range(NUM_EPOCHS):
    baseline_projector.train()
    baseline_encoder.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for win_idx in range(0, len(train_windows), BATCH_SIZE):
        batch = train_windows[win_idx:win_idx + BATCH_SIZE]
        fmri_target = torch.stack([w['data'] for w in batch]).to(device)
        
        baseline_opt.zero_grad()
        x = baseline_encoder(fmri_target)  # NO memory attention
        pred = baseline_projector(x)
        loss = F.mse_loss(pred, fmri_target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(baseline_params, 1.0)
        baseline_opt.step()
        epoch_loss += loss.item()
        n_batches += 1
    
    baseline_sched.step()
    avg = epoch_loss / max(n_batches, 1)
    baseline_losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}: loss={avg:.4f}')

# Get baseline per-parcel scores on validation
baseline_projector.eval()
baseline_encoder.eval()

baseline_preds = []
with torch.inference_mode():
    for w in val_windows:
        fmri = w['data'].unsqueeze(0).to(device)
        x = baseline_encoder(fmri)
        pred = baseline_projector(x)
        baseline_preds.append(pred.cpu().numpy())

baseline_cat = np.concatenate(baseline_preds, axis=1)[0]

baseline_r = []
for p in range(n_parcels):
    if targets_cat[:, p].std() > 1e-6 and baseline_cat[:, p].std() > 1e-6:
        r, _ = pearsonr(baseline_cat[:, p], targets_cat[:, p])
        baseline_r.append(r)
    else:
        baseline_r.append(0.0)
baseline_r = np.array(baseline_r)

print(f'\nBaseline mean R: {np.mean(baseline_r):.4f}')
print(f'Memory mean R:   {np.mean(parcel_r):.4f}')
print(f'Improvement:     {np.mean(parcel_r) - np.mean(baseline_r):.4f}')

In [ ]:
# Per-parcel improvement: memory R minus baseline R
improvement = parcel_r - baseline_r

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter: baseline vs memory per parcel
axes[0].scatter(baseline_r, parcel_r, alpha=0.3, s=10, c='blue')
lim = [min(baseline_r.min(), parcel_r.min()) - 0.05,
       max(baseline_r.max(), parcel_r.max()) + 0.05]
axes[0].plot(lim, lim, 'r--', alpha=0.5, label='y=x (no improvement)')
axes[0].set_xlabel('Baseline R (no memory)')
axes[0].set_ylabel('Memory R')
axes[0].set_title('Per-Parcel: Memory vs Baseline')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Improvement histogram
axes[1].hist(improvement, bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', label='No improvement')
axes[1].axvline(x=np.mean(improvement), color='green', linestyle='--',
               label=f'Mean={np.mean(improvement):.4f}')
axes[1].set_xlabel('R improvement (memory - baseline)')
axes[1].set_ylabel('Number of parcels')
axes[1].set_title('Memory Improvement Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Top improved vs least improved parcels
top_20 = np.argsort(improvement)[-20:]
bottom_20 = np.argsort(improvement)[:20]

y_pos = np.arange(20)
axes[2].barh(y_pos, improvement[top_20], color='green', alpha=0.7, label='Top 20 improved')
axes[2].barh(y_pos + 22, improvement[bottom_20], color='red', alpha=0.7, label='Bottom 20')
axes[2].set_xlabel('R improvement')
axes[2].set_title('Most vs Least Improved Parcels')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Memory vs No-Memory Comparison (Real fMRI)', fontsize=14, y=1.02)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'day6_memory_vs_baseline.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'\nParcels where memory helps (improvement > 0): {(improvement > 0).sum()}/{n_parcels}')
print(f'Parcels where memory hurts (improvement < 0): {(improvement < 0).sum()}/{n_parcels}')
print(f'Mean improvement: {np.mean(improvement):.4f}')

---
## 6. Save Everything

In [ ]:
import json
from datetime import datetime

# Save per-parcel scores
np.savez(
    os.path.join(RESULTS_DIR, 'day6_parcel_scores.npz'),
    memory_r=parcel_r,
    baseline_r=baseline_r,
    improvement=improvement,
)

session = {
    'date': datetime.now().isoformat(),
    'notebook': '06_real_fmri_training',
    'data': 'REAL Algonauts Subject 1 (Friends)',
    'n_episodes': len(all_fmri),
    'n_parcels': int(n_parcels),
    'train_windows': len(train_windows),
    'val_windows': len(val_windows),
    'epochs': NUM_EPOCHS,
    'final_train_loss': float(train_losses[-1]),
    'best_val_loss': float(best_val_loss),
    'final_gate': float(gate_values[-1]),
    'memory_mean_r': float(np.mean(parcel_r)),
    'baseline_mean_r': float(np.mean(baseline_r)),
    'mean_improvement': float(np.mean(improvement)),
    'parcels_improved': int((improvement > 0).sum()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'status': 'completed'
}

log_path = os.path.join(RESULTS_DIR, 'session_log.json')
logs = json.load(open(log_path)) if os.path.exists(log_path) else []
logs.append(session)
json.dump(logs, open(log_path, 'w'), indent=2)

print('Day 6 Summary — REAL fMRI DATA')
print('='*50)
print(f'Data: Subject 1, {len(all_fmri)} Friends episodes')
print(f'Train loss: {train_losses[0]:.4f} → {train_losses[-1]:.4f}')
print(f'Val loss:   {val_losses[-1]:.4f} (best: {best_val_loss:.4f})')
print(f'Gate value: {gate_values[-1]:.4f}')
print(f'Memory mean R:   {np.mean(parcel_r):.4f}')
print(f'Baseline mean R: {np.mean(baseline_r):.4f}')
print(f'Parcels improved by memory: {(improvement > 0).sum()}/{n_parcels}')
print(f'\nAll saved to Google Drive!')

---
## What You Achieved Today

For the first time, you trained on **REAL brain recordings** from a human watching Friends.

1. **Loaded actual fMRI** — 1000 brain parcels, z-scored, from Subject 1
2. **Train/val split** — proper held-out validation
3. **Memory vs baseline comparison** — per-parcel encoding scores
4. **Identified which parcels benefit from memory** — your core research result

**Next steps:**
- Map the improved parcels to brain regions using the Schaefer atlas
- Test if memory-related regions (default mode, prefrontal) show more improvement
- Scale to more episodes and more subjects
- Write up results for the paper